installation part 

In [3]:
%pip install langchain_community langchain_google_genai langchain_text_splitters langchain_core pydantic faiss-cpu pypdf python-dotenv

import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI , GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

load_dotenv() 

Note: you may need to restart the kernel to use updated packages.


True

step 1 :- chuncking 

In [4]:
docs = (
    PyPDFLoader("./documents/Company_Policies.pdf").load()
    + PyPDFLoader("./documents/Company_Profile.pdf").load()
    + PyPDFLoader("./documents/Product_and_Pricing.pdf").load()
)

In [5]:
chunks = RecursiveCharacterTextSplitter(
    chunk_size=600, chunk_overlap=150
).split_documents(docs)
print(f"Total Chunks: {len(chunks)}")

Total Chunks: 16


In [6]:
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vector_store = FAISS.from_documents(chunks, embeddings)

In [7]:
vector_store.index_to_docstore_id

{0: '4146f158-fe69-43fb-8d8a-b5a482c92041',
 1: '372f0b85-f59d-45e9-b246-193380e95237',
 2: '159c7d60-848c-4753-8a5d-74326dc9120e',
 3: '66c8c8ab-0682-4747-b205-888025f41459',
 4: 'ff5ec6df-b120-49fa-8c6f-06eb2e0d074d',
 5: '51d03894-c159-448d-b93b-3c7532103de5',
 6: '503cbbac-069d-46d5-b52f-c603196f2a77',
 7: 'b01b29e3-3464-4ac1-bbf9-17b6a8e38fab',
 8: '40948384-2210-41d2-9ce7-a324e3c3017d',
 9: 'd835df12-56e3-4558-8214-79187b64a8d9',
 10: '00fdf709-692a-4696-ba98-6f96fde5c776',
 11: '6ea7864d-24c4-4627-9a0f-8749418b21f8',
 12: '4a0a018f-6d63-45f5-98c2-8e1acaf5c913',
 13: 'bd110cac-ad9d-41cf-b835-a13bf6341a30',
 14: '64574429-f184-4403-89b8-496f320bf6f1',
 15: 'a937abaf-14b0-43d4-b821-278fe2d11b26'}

step2 :- Reterival

In [8]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [9]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x11b6f2150>, search_kwargs={'k': 3})

In [10]:
retriever.invoke('who is the CEO of the company?')

[Document(id='503cbbac-069d-46d5-b52f-c603196f2a77', metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-14T13:07:17+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-14T13:07:17+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': './documents/Company_Profile.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='Founder\nAarav Mehta founded NexaAI after over 10 years of experience in enterprise data platforms and\ncloud infrastructure.\nHe previously worked with global consulting firms where he led multiple large-scale digital\ntransformation projects.\nLeadership Team\nThe leadership team brings experience across AI engineering, product management, and business\noperations.\n\x7f\nAarav Mehta – CEO & Founder\n\x7f\nRiya Kapoor – CTO (Distributed systems & AI platforms)\n\x7f\nKunal Sharma – Head of Product\n\x7f\nNeha Verma – Head of Operations &

Step 3 - Augmentation

In [11]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [12]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [13]:
question = "tell me about the company?"
retrieved_docs    = retriever.invoke(question)

In [14]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'and education sectors.\n\x7f\nFounded: 2021\n\x7f\nHeadquarters: Bengaluru, India\n\x7f\nEmployees: 85+\n\x7f\nOperating Regions: India, Southeast Asia, Europe\nVision\nTo become a trusted AI partner for organizations by enabling intelligent decision-making through\nreliable and explainable AI systems.\nMission\nOur mission is to simplify access to organizational knowledge, automate repetitive workflows, and\nhelp teams operate with speed, clarity, and confidence.\nWe focus on practical AI adoption rather than experimental research.\nCore Values\n\nNexaAI Solutions – Company Profile\nCompany Overview\nNexaAI Solutions Pvt. Ltd. is a business-focused artificial intelligence company founded in 2021.\nThe company specializes in building enterprise-ready AI systems for knowledge management,\nanalytics, and automation.\nNexaAI primarily serves mid-sized and large organizations across technology, finance, healthcare,\nand education sectors.\n\x7f\nFounded: 2021\n\x7f\nHeadquarters: Bengalur

In [15]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

Step 4 - Generation

In [16]:
answer = llm.invoke(final_prompt)
print(answer.content)

NexaAI Solutions Pvt. Ltd. is a business-focused artificial intelligence company founded in 2021. It specializes in building enterprise-ready AI systems for knowledge management, analytics, and automation.

The company primarily serves mid-sized and large organizations across technology, finance, healthcare, and education sectors.

**Key details:**
*   **Founded:** 2021
*   **Headquarters:** Bengaluru, India
*   **Employees:** 85+
*   **Operating Regions:** India, Southeast Asia, Europe

**Vision:** To become a trusted AI partner for organizations by enabling intelligent decision-making through reliable and explainable AI systems.

**Mission:** To simplify access to organizational knowledge, automate repetitive workflows, and help teams operate with speed, clarity, and confidence, focusing on practical AI adoption rather than experimental research.

**Core Values:**
*   Customer Obsession
*   Transparency
*   Accountability
*   Security First
*   Continuous Improvement
